# EDR Threat Hunting - Isolation Forest Training

## Objective
Train an Isolation Forest model for anomaly detection on EDR telemetry features.

## Features (15 total)
- **Process (0-2)**: lineage depth, rare parent-child, privilege escalation
- **Commandline (3-5)**: length, entropy, encoded command
- **File Activity (6-8)**: modifications, sensitive access, mass activity rate
- **Network (9-11)**: connections, suspicious DNS, beaconing score
- **Persistence (12-14)**: mechanism present, cron count, service count

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import pickle

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Generate Synthetic Training Data

In [ ]:
def generate_synthetic_data(n_normal=5000, n_anomaly=500, random_state=42):
    """
    Generate synthetic EDR telemetry data.
    
    Returns:
        X: Feature matrix (n_samples, 15)
        y: Labels (1=normal, -1=anomaly)
    """
    np.random.seed(random_state)
    
    print(f"Generating {n_normal} normal + {n_anomaly} anomaly samples...")
    
    # Normal behavior patterns
    normal_data = []
    for _ in range(n_normal):
        sample = [
            np.random.randint(1, 4),        # lineage depth (shallow)
            0.0,                             # rare parent-child
            0.0,                             # privilege escalation
            np.random.randint(10, 200),     # cmdline length (short)
            np.random.uniform(2.0, 4.0),    # entropy (low)
            0.0,                             # encoded cmd
            np.random.randint(0, 5),        # file mods (few)
            0.0,                             # sensitive access
            np.random.uniform(0, 10),       # mass activity (low)
            np.random.randint(0, 3),        # connections (few)
            0.0,                             # suspicious DNS
            np.random.uniform(0, 0.3),      # beaconing (low)
            0.0,                             # persistence
            0.0,                             # cron count
            0.0,                             # service count
        ]
        normal_data.append(sample)
    
    # Anomalous behavior patterns (4 attack types)
    anomaly_data = []
    attack_types = ['encoded_powershell', 'credential_dump', 'beaconing', 'ransomware']
    
    for _ in range(n_anomaly):
        attack = np.random.choice(attack_types)
        
        if attack == 'encoded_powershell':
            sample = [
                np.random.randint(3, 6),        # Deep lineage
                1.0,                             # Rare parent-child
                np.random.choice([0.0, 1.0]),
                np.random.randint(500, 2000),   # Long cmdline
                np.random.uniform(5.0, 7.0),    # High entropy
                1.0,                             # Encoded cmd
                np.random.randint(0, 5),
                0.0,
                np.random.uniform(0, 10),
                np.random.randint(1, 5),
                np.random.choice([0.0, 1.0]),
                np.random.uniform(0, 0.5),
                0.0, 0.0, 0.0,
            ]
        elif attack == 'credential_dump':
            sample = [
                np.random.randint(2, 5),
                1.0,
                1.0,                             # Privilege escalation
                np.random.randint(100, 500),
                np.random.uniform(3.0, 5.0),
                0.0,
                np.random.randint(0, 3),
                np.random.randint(3, 10),       # Sensitive file access
                np.random.uniform(0, 5),
                np.random.randint(0, 2),
                0.0, 0.0, 0.0, 0.0, 0.0,
            ]
        elif attack == 'beaconing':
            sample = [
                np.random.randint(2, 4),
                np.random.choice([0.0, 1.0]),
                0.0,
                np.random.randint(50, 300),
                np.random.uniform(3.0, 5.0),
                0.0,
                np.random.randint(0, 3),
                0.0,
                np.random.uniform(0, 5),
                np.random.randint(10, 50),      # Many connections
                1.0,                             # Suspicious DNS
                np.random.uniform(0.8, 1.0),    # High beaconing
                np.random.choice([0.0, 1.0]),
                0.0, 0.0,
            ]
        else:  # ransomware
            sample = [
                np.random.randint(2, 4),
                np.random.choice([0.0, 1.0]),
                0.0,
                np.random.randint(100, 400),
                np.random.uniform(3.5, 5.5),
                0.0,
                np.random.randint(100, 500),    # Many file mods
                np.random.randint(0, 5),
                np.random.uniform(200, 500),    # High mass activity
                np.random.randint(1, 5),
                0.0, 0.0, 0.0, 0.0, 0.0,
            ]
        
        anomaly_data.append(sample)
    
    # Combine
    X = np.array(normal_data + anomaly_data, dtype=np.float32)
    y = np.array([1] * n_normal + [-1] * n_anomaly)
    
    # Shuffle
    indices = np.random.permutation(len(X))
    X = X[indices]
    y = y[indices]
    
    print(f"✅ Generated {len(X)} samples")
    print(f"   Normal: {(y == 1).sum()}")
    print(f"   Anomaly: {(y == -1).sum()}")
    
    return X, y

In [ ]:
# Generate data
X, y = generate_synthetic_data(n_normal=5000, n_anomaly=500)

# Convert to DataFrame for easier analysis
feature_names = [
    'lineage_depth', 'rare_parent_child', 'privilege_escalation',
    'cmdline_length', 'cmdline_entropy', 'encoded_cmd',
    'file_modifications', 'sensitive_file_access', 'mass_file_activity',
    'network_connections', 'suspicious_dns', 'beaconing_score',
    'has_persistence', 'cron_count', 'service_count'
]

df = pd.DataFrame(X, columns=feature_names)
df['label'] = y
df['label_name'] = df['label'].map({1: 'Normal', -1: 'Anomaly'})

print("\nDataset Summary:")
print(df.head())
print("\nLabel distribution:")
print(df['label_name'].value_counts())

## 2. Exploratory Data Analysis

In [ ]:
# Feature distribution comparison
fig, axes = plt.subplots(5, 3, figsize=(15, 18))
axes = axes.flatten()

for idx, feature in enumerate(feature_names):
    ax = axes[idx]
    
    normal_data = df[df['label'] == 1][feature]
    anomaly_data = df[df['label'] == -1][feature]
    
    ax.hist(normal_data, bins=30, alpha=0.5, label='Normal', color='blue')
    ax.hist(anomaly_data, bins=30, alpha=0.5, label='Anomaly', color='red')
    
    ax.set_title(feature)
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../models/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Feature distributions plotted")

In [ ]:
# Correlation matrix
plt.figure(figsize=(12, 10))
corr = df[feature_names].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../models/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Train-Test Split

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nTrain label distribution:")
print(f"  Normal: {(y_train == 1).sum()}")
print(f"  Anomaly: {(y_train == -1).sum()}")
print(f"\nTest label distribution:")
print(f"  Normal: {(y_test == 1).sum()}")
print(f"  Anomaly: {(y_test == -1).sum()}")

## 4. Train Isolation Forest Model

In [ ]:
# Train model
print("Training Isolation Forest...")

model = IsolationForest(
    n_estimators=100,
    max_samples='auto',
    contamination=0.1,  # Expected proportion of anomalies
    random_state=42,
    n_jobs=-1,
    verbose=1
)

model.fit(X_train)

print("✅ Training completed!")

## 5. Model Evaluation

In [ ]:
# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Decision scores
scores_train = model.decision_function(X_train)
scores_test = model.decision_function(X_test)

print("=" * 60)
print("MODEL EVALUATION RESULTS")
print("=" * 60)

# Training set
print("\n📊 TRAINING SET:")
print(classification_report(
    (y_train == 1).astype(int),
    (y_pred_train == 1).astype(int),
    target_names=['Anomaly', 'Normal']
))

# Test set
print("\n📊 TEST SET:")
print(classification_report(
    (y_test == 1).astype(int),
    (y_pred_test == 1).astype(int),
    target_names=['Anomaly', 'Normal']
))

In [ ]:
# Confusion Matrix
from sklearn.metrics import ConfusionMatrixDisplay

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Train
cm_train = confusion_matrix(
    (y_train == 1).astype(int),
    (y_pred_train == 1).astype(int)
)
disp1 = ConfusionMatrixDisplay(cm_train, display_labels=['Anomaly', 'Normal'])
disp1.plot(ax=ax1, cmap='Blues')
ax1.set_title('Confusion Matrix - Training Set')

# Test
cm_test = confusion_matrix(
    (y_test == 1).astype(int),
    (y_pred_test == 1).astype(int)
)
disp2 = ConfusionMatrixDisplay(cm_test, display_labels=['Anomaly', 'Normal'])
disp2.plot(ax=ax2, cmap='Blues')
ax2.set_title('Confusion Matrix - Test Set')

plt.tight_layout()
plt.savefig('../models/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC AUC Score
try:
    roc_auc_train = roc_auc_score((y_train == 1).astype(int), -scores_train)
    roc_auc_test = roc_auc_score((y_test == 1).astype(int), -scores_test)
    
    print(f"\n📈 ROC AUC Scores:")
    print(f"   Training: {roc_auc_train:.4f}")
    print(f"   Test:     {roc_auc_test:.4f}")
except Exception as e:
    print(f"⚠️  Could not calculate ROC AUC: {e}")

In [ ]:
# Anomaly Score Distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(scores_test[y_test == 1], bins=50, alpha=0.5, label='Normal', color='blue')
plt.hist(scores_test[y_test == -1], bins=50, alpha=0.5, label='Anomaly', color='red')
plt.xlabel('Anomaly Score')
plt.ylabel('Frequency')
plt.title('Anomaly Score Distribution (Test Set)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(
    [scores_test[y_test == 1], scores_test[y_test == -1]],
    labels=['Normal', 'Anomaly']
)
plt.ylabel('Anomaly Score')
plt.title('Anomaly Score Distribution (Boxplot)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../models/anomaly_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Model

In [ ]:
# Save model
import os

os.makedirs('../models', exist_ok=True)
model_path = '../models/isolation_forest.pkl'

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f"✅ Model saved: {model_path}")
print(f"   Size: {os.path.getsize(model_path) / 1024:.2f} KB")

## 7. Test Inference

In [ ]:
# Test with sample inputs
print("Testing model inference with sample data...\n")

# Normal behavior
normal_sample = np.array([[
    2, 0, 0,        # Process: shallow lineage, normal parent-child, no escalation
    50, 3.2, 0,     # Cmdline: short, low entropy, not encoded
    2, 0, 5,        # File: few mods, no sensitive, low mass activity
    1, 0, 0.1,      # Network: few connections, no suspicious DNS, no beaconing
    0, 0, 0         # Persistence: none
]], dtype=np.float32)

pred_normal = model.predict(normal_sample)
score_normal = model.decision_function(normal_sample)
print(f"Normal behavior:")
print(f"  Prediction: {'Normal' if pred_normal[0] == 1 else 'Anomaly'}")
print(f"  Score: {score_normal[0]:.4f}\n")

# Malicious behavior (encoded PowerShell)
malicious_sample = np.array([[
    5, 1, 1,        # Process: deep lineage, rare parent-child, privilege escalation
    800, 6.5, 1,    # Cmdline: long, high entropy, encoded
    3, 2, 8,        # File: some mods, sensitive access, moderate activity
    4, 1, 0.4,      # Network: several connections, suspicious DNS
    0, 0, 0         # Persistence: none
]], dtype=np.float32)

pred_malicious = model.predict(malicious_sample)
score_malicious = model.decision_function(malicious_sample)
print(f"Malicious behavior (encoded PowerShell):")
print(f"  Prediction: {'Normal' if pred_malicious[0] == 1 else 'Anomaly'}")
print(f"  Score: {score_malicious[0]:.4f}\n")

print("✅ Inference test complete!")

## Summary

The Isolation Forest model has been trained and evaluated. Next steps:

1. Export to ONNX format: Run `export_onnx.py` script
2. Deploy to K8s: Copy model to agent pods
3. Monitor performance: Check Grafana dashboards

**Model is ready for deployment!** 🚀